
# VeriPulse GRAPE → GRAPE_AVG/SI warm-start smoke test

This notebook tests the workflow:

1. Take a **non-AVG GRAPE** result whose `final_amps` have shape `(K, num_tslots, 2)`.
2. Convert those amplitudes into the packed control shape expected by `run_grape_si`, namely `(num_tslots, 2*K)`.
3. Run dimension checks before using the pulse as the initial pulse for a small `SECRETIND` / `GRAPE_AVG` smoke test.
4. Recompute secret independence using `choi_optimise_secret_indep()` and test the returned Choi matrix.

The key subtlety is the control ordering. In `nvcenter_system(K)`, the multi-qubit controls are created as all `x` controls first, followed by all `y` controls:

```python
L_ctrl = [x0, x1, ..., x_{K-1}, y0, y1, ..., y_{K-1}]
```

Therefore the warm-start conversion used here is **block ordered**, not interleaved.



## Environment

Run this notebook from the root of the VeriPulse repository, or place it in `notebooks/` and let the path helper below find `../src`.

The repository README says to use `uv` and the forked `qutip-qtrl` submodule:

```bash
git clone --recurse-submodules https://github.com/cicacica/VeriPulse.git
cd VeriPulse
uv sync
uv pip install -e vendor/qutip-qtrl
uv run python -m ipykernel install --user --name veripulse --display-name "VeriPulse"
uv run jupyter lab
```

Then select the `VeriPulse` kernel.


In [ ]:

from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np

# Find the repository root whether this notebook is in repo root or in notebooks/.
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "src" / "veripulse").exists():
            return p
    raise RuntimeError(
        "Could not find repo root containing src/veripulse. "
        "Run this notebook from the VeriPulse repo or put it under VeriPulse/notebooks/."
    )

REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("REPO_ROOT =", REPO_ROOT)
print("SRC       =", SRC)


In [ ]:

from numpy import pi, stack

from veripulse.pulse import (
    PulseConfig,
    PulseResult,
    run_grape,
    run_grape_si,
    nvcenter_system,
)
from veripulse.gates import (
    Qobj,
    operator_to_vector,
    pack_subspace_states,
    rx,
    rhox,
    hadamard,
    hadamardZ,
    extract_subspace_states,
)
from veripulse.sdp import (
    choi_optimise_secret_indep,
    calc_secret_indep,
)

print("Imports OK")



## Test parameters

By default, this uses only the first two states from the saved non-AVG GRAPE result, so the SI smoke test is small. Increase `K_TEST` to use more of the saved pulse.


In [ ]:

# Use a subset of the uploaded/saved non-AVG GRAPE pulse for a small smoke test.
K_TEST = 2
LAM = 100.0
RUN_SI_OPTIMIZATION = True  # set False if you only want shape checks
SI_MAX_ITER = 1             # one iteration is enough to verify warm-start initialisation

# Candidate paths for your non-AVG GRAPE result JSON.
# Put GRAPE_p70_det0.00_err0.00-1.json next to this notebook, or leave it in data/dummyyes/.
GRAPE_JSON_CANDIDATES = [
    Path.cwd() / "GRAPE_p70_det0.00_err0.00-1.json",
    REPO_ROOT / "GRAPE_p70_det0.00_err0.00-1.json",
    REPO_ROOT / "data" / "dummyyes" / "GRAPE_p70_det0.00_err0.00-1.json",
    Path("/mnt/data/Pasted text (2).txt"),  # works only in this ChatGPT sandbox
]


In [ ]:

def load_first_existing_json(paths: list[Path]) -> tuple[dict, Path]:
    checked = []
    for p in paths:
        checked.append(str(p))
        if p.exists():
            with p.open("r", encoding="utf-8") as f:
                return json.load(f), p
    raise FileNotFoundError(
        "Could not find a non-AVG GRAPE result JSON. Checked:\n" + "\n".join(checked)
    )

nonavg_data, nonavg_path = load_first_existing_json(GRAPE_JSON_CANDIDATES)
print("Loaded:", nonavg_path)
print("label:", nonavg_data.get("label"))
print("mode :", nonavg_data.get("mode"))

amps_nonavg_all = np.asarray(nonavg_data["final_amps"], dtype=float)
labels_all = nonavg_data.get("state_labels", list(range(amps_nonavg_all.shape[0])))
config_data = nonavg_data.get("config", {})

print("saved final_amps shape:", amps_nonavg_all.shape)
print("number of labels       :", len(labels_all))


In [ ]:

def pack_grape_amps_for_si_block_order(amps_by_state: np.ndarray) -> np.ndarray:
    """
    Convert non-AVG GRAPE amplitudes to the multi-control SI/GRAPE_AVG warm-start shape.

    Input shape:
        (K, N, 2), where amps_by_state[j, :, 0] is x_j(t) and amps_by_state[j, :, 1] is y_j(t).

    Output shape:
        (N, 2*K), ordered as [x0, x1, ..., x_{K-1}, y0, y1, ..., y_{K-1}].

    This matches nvcenter_system(K), where L_ctrl is built as all x controls followed by all y controls.
    """
    amps_by_state = np.asarray(amps_by_state, dtype=float)
    if amps_by_state.ndim != 3:
        raise ValueError(f"Expected 3D array (K, N, 2), got shape {amps_by_state.shape}")
    K, N, num_single_controls = amps_by_state.shape
    if num_single_controls != 2:
        raise ValueError(f"Expected the last dimension to be 2 for x/y controls, got {num_single_controls}")

    x_block = amps_by_state[:, :, 0].T  # (N, K) = [x0, x1, ...]
    y_block = amps_by_state[:, :, 1].T  # (N, K) = [y0, y1, ...]
    return np.hstack([x_block, y_block])


def pack_grape_amps_interleaved_for_plotting_only(amps_by_state: np.ndarray) -> np.ndarray:
    """
    Alternative layout [x0, y0, x1, y1, ...].
    This is useful for some plotting code, but it does NOT match nvcenter_system(K)'s current L_ctrl order.
    """
    amps_by_state = np.asarray(amps_by_state, dtype=float)
    return amps_by_state.transpose(1, 0, 2).reshape(amps_by_state.shape[1], 2 * amps_by_state.shape[0])


In [ ]:

# Keep the test small by slicing the saved non-AVG GRAPE result.
K_TOTAL, N_TSLOTS, N_SINGLE_CTRLS = amps_nonavg_all.shape
assert nonavg_data["mode"] == "GRAPE", f"Expected a non-AVG GRAPE result, got mode={nonavg_data['mode']!r}"
assert N_SINGLE_CTRLS == 2, f"Expected 2 single-qubit controls, got {N_SINGLE_CTRLS}"
assert len(labels_all) == K_TOTAL, "state_labels length must match final_amps first dimension"
assert 1 <= K_TEST <= K_TOTAL, f"K_TEST must be in [1, {K_TOTAL}]"

amps_nonavg = amps_nonavg_all[:K_TEST]
labels = labels_all[:K_TEST]

# Reconstruct the same state convention as run_sampling_data.py / your earlier script.
# Numeric labels are angles for rx/rhox; '+' and '-' are the dummy Hadamard targets.
rho_init = Qobj([[1.0, 0.0], [0.0, 0.0]])
rho_targets = []
unitary_rotations = []

for label in labels:
    if label == "+":
        rho_targets.append(Qobj([[0.5, 0.5], [0.5, 0.5]]))
        unitary_rotations.append(hadamard())
    elif label == "-":
        rho_targets.append(Qobj([[0.5, -0.5], [-0.5, 0.5]]))
        unitary_rotations.append(hadamardZ())
    else:
        angle = float(label)
        rho_targets.append(Qobj(rhox(angle)))
        unitary_rotations.append(rx(angle))

vRho_init, vRho_target, U_big = pack_subspace_states(
    rotations=unitary_rotations,
    rho_init=rho_init,
)

D = int(np.sqrt(np.shape(vRho_init)[0]))
K_from_packed_state = D // 2
assert K_from_packed_state == K_TEST, (K_from_packed_state, K_TEST)

print("labels used:", labels)
print("K_TEST:", K_TEST)
print("vRho_init shape  :", np.shape(vRho_init))
print("vRho_target shape:", np.shape(vRho_target))
print("U_big shape      :", np.shape(U_big))


In [ ]:

# Reuse the saved GRAPE config, but make the SI smoke test cheap.
cfg = PulseConfig(
    omega_drift=config_data.get("omega_drift", 10e6),
    T2_star=config_data.get("T2_star", 2e-6),
    drive_error=config_data.get("drive_error", 0.0),
    detuning=config_data.get("detuning", 0.0),
    evo_time=config_data.get("evo_time", 75e-9),
    num_tslots=N_TSLOTS,
    amp_lbound=config_data.get("amp_lbound", -1.0),
    amp_ubound=config_data.get("amp_ubound", 1.0),
    awg_resolution=config_data.get("awg_resolution", 15e-12),
    fid_err_targ=1e-6,
    max_iter=SI_MAX_ITER,
    max_wall_time=60,
    init_pulse_type="RND",
)

L_drift, L_ctrl = nvcenter_system(K_TEST, cfg)
expected_num_ctrls = len(L_ctrl)
expected_shape = (cfg.num_tslots, expected_num_ctrls)

amps_si = pack_grape_amps_for_si_block_order(amps_nonavg)
amps_interleaved = pack_grape_amps_interleaved_for_plotting_only(amps_nonavg)

print("non-AVG sliced shape          :", amps_nonavg.shape)
print("SI warm-start shape           :", amps_si.shape)
print("interleaved plotting shape    :", amps_interleaved.shape)
print("expected SI shape from L_ctrl :", expected_shape)

assert amps_si.shape == expected_shape, (amps_si.shape, expected_shape)
assert np.all(np.isfinite(amps_si)), "Warm-start contains NaN or inf"
assert np.min(amps_si) >= cfg.amp_lbound - 1e-12, (np.min(amps_si), cfg.amp_lbound)
assert np.max(amps_si) <= cfg.amp_ubound + 1e-12, (np.max(amps_si), cfg.amp_ubound)
assert len(rho_targets) == K_TEST
assert len(unitary_rotations) == K_TEST
assert np.shape(U_big)[0] == 2*K_TEST and np.shape(U_big)[1] == 2*K_TEST

print("All dimension and amplitude-bound checks passed.")



## Run the SI warm-start smoke test

With `SI_MAX_ITER = 1`, this is not meant to produce a good optimum. It only verifies that the packed non-AVG GRAPE pulse can be used as `amps=` in `run_grape_si` without a shape/order error.


In [ ]:

if RUN_SI_OPTIMIZATION:
    result_si = run_grape_si(
        vRho_init,
        vRho_target,
        U_big,
        lam=LAM,
        config=cfg,
        amps=amps_si,
    )

    print("SI smoke test finished.")
    print("fid_err:", result_si.fid_err)
    print("termination:", result_si.termination_reason)
    print("result.final_amps shape:", result_si.final_amps.shape)
    assert result_si.final_amps.shape == expected_shape
else:
    print("RUN_SI_OPTIMIZATION=False, so only the shape checks were run.")



## Recalculate secret independence with the Choi SDP

The cells below recompute the SI value explicitly with `choi_optimise_secret_indep()`. They also test the SDP output:

- target/final state list lengths and matrix dimensions,
- finite objective value,
- valid Choi shape `(4, 4)`,
- Hermiticity and positive semidefiniteness of the Choi matrix,
- trace-preserving constraint `Tr_B J = I`,
- consistency between the SDP objective and `calc_secret_indep(..., J=choi)`.


In [ ]:

def matrix_from_split(real_part, imag_part) -> np.ndarray:
    """Reconstruct a complex matrix stored as separate real/imag JSON arrays."""
    return np.asarray(real_part, dtype=float) + 1j * np.asarray(imag_part, dtype=float)


def as_2x2_density_list(mats, name: str, *, tol: float = 5e-6) -> list[np.ndarray]:
    """Convert to complex 2x2 matrices and run lightweight density-matrix sanity checks."""
    out = []
    for k, mat in enumerate(mats):
        arr = np.asarray(mat, dtype=complex)
        assert arr.shape == (2, 2), f"{name}[{k}] has shape {arr.shape}, expected (2, 2)"
        assert np.all(np.isfinite(arr.real)) and np.all(np.isfinite(arr.imag)), f"{name}[{k}] is not finite"
        assert np.allclose(arr, arr.conj().T, atol=tol), f"{name}[{k}] is not Hermitian"
        assert np.isclose(np.trace(arr), 1.0, atol=tol), f"{name}[{k}] trace is {np.trace(arr)}"
        evals = np.linalg.eigvalsh((arr + arr.conj().T) / 2)
        assert np.min(evals) >= -1e-4, f"{name}[{k}] has negative eigenvalue {np.min(evals)}"
        out.append(arr)
    return out


def partial_trace_output_of_choi(J: np.ndarray) -> np.ndarray:
    """
    Compute Tr_B J using the same convention as veripulse.sdp.choi_optimise_secret_indep.
    For a valid trace-preserving channel, this should be the 2x2 identity.
    """
    e = np.eye(2)
    return sum(
        (np.kron(np.eye(2), e[b].reshape(1, 2)))
        @ J
        @ (np.kron(np.eye(2), e[b].reshape(2, 1)))
        for b in range(2)
    )


def run_choi_si_tests(
    rho_targ: list[np.ndarray],
    rho_ests: list[np.ndarray],
    *,
    label: str,
    solver_opts: dict | None = None,
    tol: float = 5e-4,
) -> dict:
    """
    Recompute SI using choi_optimise_secret_indep and verify the returned SDP solution.
    """
    assert len(rho_targ) == len(rho_ests), "rho_targ and rho_ests must have the same length"
    assert len(rho_targ) >= 1, "Need at least one target/estimate pair"

    rho_targ = as_2x2_density_list(rho_targ, f"{label}.rho_targ")
    rho_ests = as_2x2_density_list(rho_ests, f"{label}.rho_ests")

    # Baseline: SI without any correcting channel.
    si_no_channel = float(calc_secret_indep(rho_targ, rho_ests))
    assert np.isfinite(si_no_channel) and si_no_channel >= -tol

    # SDP: optimise over a CPTP Choi matrix.
    solver_opts = solver_opts or {"verbose": False, "eps": 1e-6, "max_iters": 5000}
    res = choi_optimise_secret_indep(rho_targ, rho_ests, solver_opts=solver_opts)

    assert res.status in {"optimal", "optimal_inaccurate"}, f"Unexpected SDP status: {res.status}"
    assert res.objective is not None and np.isfinite(res.objective), "SDP objective is not finite"
    assert res.objective >= -tol, f"SDP objective is negative beyond tolerance: {res.objective}"

    J = np.asarray(res.choi, dtype=complex)
    assert J.shape == (4, 4), f"Choi matrix has shape {J.shape}, expected (4, 4)"
    assert np.all(np.isfinite(J.real)) and np.all(np.isfinite(J.imag)), "Choi matrix is not finite"
    assert np.allclose(J, J.conj().T, atol=1e-5), "Choi matrix is not Hermitian"

    choi_evals = np.linalg.eigvalsh((J + J.conj().T) / 2)
    assert np.min(choi_evals) >= -1e-4, f"Choi matrix is not PSD: min eigenvalue {np.min(choi_evals)}"

    tr_B = partial_trace_output_of_choi(J)
    assert np.allclose(tr_B, np.eye(2), atol=2e-4), f"Trace-preserving check failed: Tr_B(J)={tr_B}"

    # Manual recomputation using the optimised Choi matrix.
    si_from_choi = float(calc_secret_indep(rho_targ, rho_ests, J=J))
    assert np.isfinite(si_from_choi) and si_from_choi >= -tol
    assert abs(si_from_choi - float(res.objective)) <= max(tol, 5e-2 * max(1.0, abs(float(res.objective)))), (
        si_from_choi,
        res.objective,
    )

    # Optimising over channels should not be worse than using no correcting channel.
    assert float(res.objective) <= si_no_channel + 5 * tol, (res.objective, si_no_channel)

    print(f"[{label}] status          :", res.status)
    print(f"[{label}] SI no channel   : {si_no_channel:.6e}")
    print(f"[{label}] SI SDP objective: {float(res.objective):.6e}")
    print(f"[{label}] SI from Choi    : {si_from_choi:.6e}")
    print(f"[{label}] min eig(Choi)   : {np.min(choi_evals):.6e}")
    print(f"[{label}] max |Tr_B J-I|  : {np.max(np.abs(tr_B - np.eye(2))):.6e}")

    return {
        "result": res,
        "si_no_channel": si_no_channel,
        "si_from_choi": si_from_choi,
        "choi_min_eig": float(np.min(choi_evals)),
        "trace_preserving_error": float(np.max(np.abs(tr_B - np.eye(2)))),
    }


In [ ]:

# Always test the SDP machinery on the loaded non-AVG GRAPE subset.
# This works even if RUN_SI_OPTIMIZATION=False.
rho_targets_loaded_all = [
    matrix_from_split(re, im)
    for re, im in zip(nonavg_data["rho_targets_re"], nonavg_data["rho_targets_im"])
]
rho_finals_loaded_all = [
    matrix_from_split(re, im)
    for re, im in zip(nonavg_data["rho_finals_re"], nonavg_data["rho_finals_im"])
]

rho_targets_loaded = rho_targets_loaded_all[:K_TEST]
rho_finals_loaded = rho_finals_loaded_all[:K_TEST]

si_loaded_checks = run_choi_si_tests(
    rho_targets_loaded,
    rho_finals_loaded,
    label=f"loaded_nonavg_subset_K{K_TEST}",
)

# Only compare with the saved SI if the whole saved dataset is being used.
if K_TEST == K_TOTAL and nonavg_data.get("si") is not None:
    saved_si = float(nonavg_data["si"])
    recomputed_si = float(si_loaded_checks["result"].objective)
    print("saved SI from JSON       :", f"{saved_si:.6e}")
    print("recomputed SI from SDP   :", f"{recomputed_si:.6e}")
    assert np.isclose(recomputed_si, saved_si, rtol=5e-2, atol=5e-4), (recomputed_si, saved_si)
else:
    print("Skipping saved-SI comparison because K_TEST != K_TOTAL or saved SI is absent.")


In [ ]:

# Recompute SI for the actual SI/GRAPE_AVG smoke-test result, if it was run.
if "result_si" in globals():
    rho_targets_np = [rho.full() for rho in rho_targets]
    rho_finals_si = extract_subspace_states(result_si, K_TEST)

    assert len(rho_finals_si) == K_TEST
    assert result_si.final_amps.shape == expected_shape

    si_result_checks = run_choi_si_tests(
        rho_targets_np,
        rho_finals_si,
        label="run_grape_si_result",
    )

    print("run_grape_si result fid_err:", result_si.fid_err)
else:
    print("No result_si found. Run the SI smoke-test cell above, or set RUN_SI_OPTIMIZATION=True.")



## Optional: generate a fresh tiny non-AVG GRAPE result

Run this cell only if you want to create a fresh non-AVG GRAPE warm-start inside the notebook. This is slower than loading a saved JSON, but it gives a fully self-contained test for a few states.


In [ ]:

RUN_FRESH_NONAVG_GRAPE = False

if RUN_FRESH_NONAVG_GRAPE:
    fresh_cfg = PulseConfig(
        num_tslots=20,
        max_iter=5,
        max_wall_time=60,
        fid_err_targ=1e-4,
        init_pulse_type="RND",
    )
    fresh_angles = [0.0, pi / 4]
    fresh_rho_targets = [Qobj(rhox(a)) for a in fresh_angles]
    fresh_rotations = [rx(a) for a in fresh_angles]
    fresh_rho_init = Qobj([[1.0, 0.0], [0.0, 0.0]])

    fresh_results = []
    for rho_targ in fresh_rho_targets:
        fresh_results.append(
            run_grape(
                operator_to_vector(fresh_rho_init),
                operator_to_vector(rho_targ),
                config=fresh_cfg,
            )
        )

    fresh_amps_nonavg = stack([r.final_amps for r in fresh_results])
    fresh_amps_si = pack_grape_amps_for_si_block_order(fresh_amps_nonavg)

    fresh_vRho_init, fresh_vRho_target, fresh_U_big = pack_subspace_states(
        rotations=fresh_rotations,
        rho_init=fresh_rho_init,
    )

    _, fresh_L_ctrl = nvcenter_system(len(fresh_angles), fresh_cfg)
    assert fresh_amps_si.shape == (fresh_cfg.num_tslots, len(fresh_L_ctrl))

    fresh_si = run_grape_si(
        fresh_vRho_init,
        fresh_vRho_target,
        fresh_U_big,
        lam=LAM,
        config=fresh_cfg,
        amps=fresh_amps_si,
    )

    print("Fresh non-AVG → SI smoke test finished.")
    print("fresh non-AVG shape:", fresh_amps_nonavg.shape)
    print("fresh SI warm-start shape:", fresh_amps_si.shape)
    print("fresh SI final_amps shape:", fresh_si.final_amps.shape)
